In [1]:
%pip install -q faster-whisper pyannote.audio==4.0.0
%pip install -q spacy
%pip install groq
%pip install reportlab pandas
%pip install flask
%pip install python-dotenv
%pip install flask-cors

In [2]:
%pip install spacy download en_core_web_sm

In [3]:
from google.colab import drive
from dotenv import load_dotenv
import os

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [4]:
# Load .env from Drive
load_dotenv('/content/drive/MyDrive/SMMG/.env')

# Verify
HF_token = os.getenv("HF_token")
Gemini_key = os.getenv("Gemini_key")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
# =====================================================ASR========================================================

import torch
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
import os
import gc
from torch.torch_version import TorchVersion
from pyannote.audio.core.task import Specifications, Problem, Resolution
import subprocess
import base64
# =========================
# SETUP
# =========================

# Pyannote safety globals
torch.serialization.add_safe_globals([TorchVersion, Specifications, Problem, Resolution])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# HELPER FUNCTIONS
# =========================

def cleanup():
    torch.cuda.empty_cache()
    gc.collect()

def convert_to_wav(input_path):
    """
    Convert any audio file to .wav format using ffmpeg
    """
    output_path = input_path.rsplit(".", 1)[0] + ".wav"

    if input_path.endswith(".wav"):
        return input_path

    command = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        output_path
    ]

    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path

# =========================
# LOAD MODELS
# =========================

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_token
).to(device)

model = WhisperModel(
    "medium",
    device="cuda",
    compute_type="float16"
)

# =========================
# CORE FUNCTIONS
# =========================

def run_diarization(audio_path):
    diarization = diarization_pipeline(audio_path)
    cleanup()
    return diarization


def run_transcription(audio_path, domain_prompt):
    print(f"Running Whisper with prompt: '{domain_prompt}'")

    segments, _ = model.transcribe(
        audio_path,
        language="en",
        beam_size=5,
        vad_filter=True,
        initial_prompt=domain_prompt
    )

    asr_segments = [
        {"start": s.start, "end": s.end, "text": s.text.strip()}
        for s in segments
    ]

    cleanup()
    return asr_segments


def merge_results(diarization, asr_segments):
    """
    Merge Whisper ASR with diarization (speaker mapping)
    """

    raw_transcript = ""

    if hasattr(diarization, "speaker_diarization"):
        annotation = diarization.speaker_diarization
    else:
        annotation = diarization

    speaker_turns = [
        (turn.start, turn.end, speaker)
        for turn, _, speaker in annotation.itertracks(yield_label=True)
    ]

    for seg in asr_segments:
        seg_start = seg["start"]
        seg_end = seg["end"]

        max_overlap = 0
        assigned_speaker = "Unknown"

        for start, end, speaker in speaker_turns:
            overlap = min(seg_end, end) - max(seg_start, start)

            if overlap > max_overlap:
                max_overlap = overlap
                assigned_speaker = speaker

        timestamp = f"[{seg_start:.2f}s - {seg_end:.2f}s]"
        raw_transcript += f"{timestamp} {assigned_speaker}: {seg['text']}\n"

    return raw_transcript

# =========================
# MAIN FUNCTION (IMPORTANT)
# =========================

def stt_pipeline(audio_path, domain_prompt=None):
    """
    FULL STT PIPELINE (single file)

    Input:
        audio_path → path of uploaded audio file

    Output:
        raw_transcript → string
    """

    # Convert audio to wav if needed
    audio_path = convert_to_wav(audio_path)

    if not domain_prompt:
        domain_prompt = "Budget meeting, IT software, product development, technical discussion"

    try:
        diarization_out = run_diarization(audio_path)
        asr_out = run_transcription(audio_path, domain_prompt)

        raw_transcript = merge_results(diarization_out, asr_out)

        print("STT completed successfully")
        return raw_transcript

    except Exception as e:
        print(f"STT FAILED: {str(e)}")
        return ""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

In [6]:
# ==================================================PREPROCESSING============================================

import re
import spacy

# Load once
nlp = spacy.load("en_core_web_sm", disable=["ner"])

# =========================
# CLEANING FUNCTIONS
# =========================

def remove_timestamps(text):
    text = re.sub(r'\[\d+(\.\d+)?s\s*-\s*\d+(\.\d+)?s\]', '', text)
    text = re.sub(r'\[?\d{1,2}:\d{2}(:\d{2})?\]?', '', text)
    text = re.sub(r'\(\d+(\.\d+)?\)', '', text)
    text = re.sub(r'\d{1,2}:\d{2}:\d{2}\s*(AM|PM)?', '', text)
    return text

def remove_annotations(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\{[^}]*\}', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\b\w+-\s', ' ', text)
    text = re.sub(r'%\w+', '', text)
    return text

def remove_fillers(text):
    fillers = (
        r'\b(um+|uh+|hmm+|hm+|mhm+|uh-huh|mm+|'
        r'you know|kind of|sort of|i mean|'
        r'basically|actually|literally|'
        r'right\?|okay\?|you see|you get me)\b'
    )
    text = re.sub(fillers, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_speaker_and_text(line):
    line = re.sub(r'^>+\s*', '', line.strip())

    bracket = re.match(r'^\[([^\]]+)\]\s*(.*)', line)
    if bracket:
        return bracket.group(1).strip(), bracket.group(2).strip()

    colon = re.match(r'^([A-Za-z][A-Za-z0-9_ ]{0,30})\s*:\s*(.*)', line)
    if colon:
        return colon.group(1).strip(), colon.group(2).strip()

    return None, None

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def segment_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

def is_valid_sentence(sentence):
    words = sentence.split()
    if len(words) < 5:
        return False

    has_verb = re.search(
        r'\b(is|are|was|were|be|been|being|'
        r'do|does|did|done|have|has|had|'
        r'will|shall|should|can|could|may|might|must|would|'
        r'need|needs|needed|let|lets|'
        r'going|plan|plans|planned|want|wants|wanted|'
        r'agree|agreed|agrees|decide|decided|decides|'
        r'propose|proposed|proposes|think|thinks|thought|'
        r'suggest|suggests|suggested|make|makes|made|'
        r'take|takes|took|send|sends|sent|review|reviews|reviewed|'
        r'check|checks|checked|update|updates|updated|'
        r'assign|assigns|assigned|schedule|schedules|scheduled|'
        r'follow|follows|followed|present|presents|presented|'
        r'discuss|discusses|discussed|confirm|confirms|confirmed|'
        r'approve|approves|approved)\b',
        sentence, flags=re.IGNORECASE
    )

    if not has_verb:
        return False

    noise_only = re.compile(
        r'^(okay|ok|right|yeah|yes|no|sure|alright|fine|good|'
        r'great|perfect|absolutely|exactly|indeed|'
        r'thank you|thanks|welcome|sorry|excuse me|'
        r'i see|i know|got it|makes sense)[.!?,\s]*$',
        re.IGNORECASE
    )

    if noise_only.match(sentence.strip()):
        return False

    return True

# =========================
# MAIN FUNCTION (IMPORTANT)
# =========================

def preprocess_pipeline(raw_transcript):
    """
    Input:
        raw_transcript (string from STT)

    Output:
        cleaned_text (string)
    """

    raw_text = remove_timestamps(raw_transcript)
    raw_text = remove_annotations(raw_text)

    lines_to_write = []

    for line in raw_text.split("\n"):

        line = normalize_whitespace(line)
        if not line:
            continue

        speaker, content = extract_speaker_and_text(line)
        if speaker is None:
            continue

        content = remove_fillers(content)
        if not content:
            continue

        sentences = segment_sentences(content)

        for sentence in sentences:
            if not is_valid_sentence(sentence):
                continue

            lines_to_write.append(f"{speaker}: {sentence}")

    if not lines_to_write:
        return ""

    return "\n".join(lines_to_write)

In [8]:
#=========================================LABELLING=============================================
from groq import Groq
import time
import re

# =========================
# SETUP
# =========================

api_key = GROQ_API_KEY
client = Groq(api_key=api_key)

# =========================
# SYSTEM PROMPT
# =========================

SYSTEM_PROMPT = """You are an expert classifier for professional meeting transcripts.
Your task is to read each sentence and assign it exactly ONE label from: Action, Decision, Proposal, Discussion.

ACTION — A concrete, specific task that someone is assigned to or commits to doing.
  Requirements:
    • Must have a clear WHAT (specific deliverable or task)
    • Must have a clear WHO (named person or "I")
    • Bonus if it has WHEN (deadline), but not required
  Trigger phrases: "I will send", "I'll schedule", "can you review", "please confirm", "I'll prepare", "I will follow up"
  ✓ REAL Actions:
    "I'll send the updated report to the team by Friday"
    "Can you review the design mockups before Thursday"
    "I will set up the client call for next week"
    "Speaker_02 please confirm the budget numbers"
  ✗ NOT Actions (→ Discussion):
    "I'll look into it" — too vague, no specific deliverable
    "I'll handle it" — vague, no clear task
    "I think you should check" — opinion, not a commitment
    "I can help with that" — offer, not a commitment
    "I'll show you later" — vague, no deliverable

DECISION — A confirmed, agreed-upon outcome that affects the project or team direction.
  Requirements:
    • Must be FINALIZED — not just suggested or considered
    • Must be AGREED by the group or authority
    • Changes something about the project, timeline, scope, or approach
  Trigger phrases: "we have decided", "we agreed", "it is confirmed", "we will not", "we are going with", "the decision is"
  ✓ REAL Decisions:
    "We have decided to use React instead of Angular"
    "We confirmed the release date is March 15th"
    "We will not be supporting that feature in v1"
    "The team agreed to push the deadline by one week"
  ✗ NOT Decisions (→ Proposal or Discussion):
    "We might go with React" — uncertain, not confirmed
    "Let us use React" — suggestion, not confirmed decision
    "Yeah that makes sense" — agreement to a point, not a project decision
    "We should finalize this" — talking about deciding, not a decision itself

PROPOSAL — A specific suggestion being raised for group consideration, not yet decided.
  Requirements:
    • Must be about a WORK-RELATED topic
    • Must be a suggestion, recommendation, or idea put forward
    • Group has NOT yet agreed on it
  Trigger phrases: "we should", "we could", "I suggest", "what if we", "maybe we", "I recommend", "how about", "let's consider"
  ✓ REAL Proposals:
    "We should move the meeting to Thursday"
    "I suggest we break this into two separate tasks"
    "What if we use a different vendor for this module"
    "Maybe we could reduce the scope for the first release"
  ✗ NOT Proposals (→ Discussion):
    "What do you think about this approach" — question, not a proposal
    "That could be an option" — vague acknowledgment
    Social or personal suggestions ("we should grab lunch")

DISCUSSION — Everything that does not fit Action, Decision, or Proposal.
  This includes:
    • Questions and follow-up questions
    • Reactions, agreements, and acknowledgments ("sounds good", "that makes sense", "exactly")
    • Status updates about what already happened ("I sent the email yesterday")
    • Explanations, context-setting, and background information
    • Vague commitments without a clear deliverable ("I'll look into it", "I'll think about it")
    • Social conversation, jokes, greetings, compliments
    • Emotional reactions ("I can't wait", "that's exciting", "great job")
    • Scheduling logistics and calendar talk
    • Meta-comments about the meeting itself ("let's move on", "let's discuss this")
    • Incomplete or ambiguous thoughts

1. "I'll / I will" is Action ONLY when followed by a SPECIFIC, CONCRETE task verb
   (send, schedule, review, prepare, write, submit, build, fix, deploy, call, email, share, create, confirm, draft, test, coordinate)
   If it is followed by a vague word (handle, look into, think about, check, try, see), classify as Discussion.

2. "We should / we could / maybe we" = ALWAYS Proposal, NEVER Decision.
   A Decision requires past confirmation, not future suggestion.

3. Questions are ALWAYS Discussion — even if they sound like task assignments.
   Exception: "Can you [concrete verb]?" directed at a named person = Action.

4. Past tense statements about completed work = Discussion.
   ("I sent it", "I already checked", "we finished that")

5. Reactions and acknowledgments = Discussion.
   ("sounds good", "exactly", "that's right", "I agree", "makes sense", "sure")

6. If a sentence is about PERSONAL or SOCIAL topics = Discussion.

7. Scheduling talk (picking meeting times, calendar availability) = Discussion UNLESS
   it results in a confirmed final decision ("the next sync is confirmed for Tuesday at 3pm" = Decision).

8. WHEN IN DOUBT → Discussion. It is better to under-label than over-label.

Return ONLY a numbered list of labels. No explanations. No extra text. No blank lines between labels.
1. Label
2. Label
3. Label
"""

# =========================
# LLM CALL
# (uses llama-3.1-8b-instant — 500k TPD vs 100k for 70b)
# =========================

def classify_batch(sentences, batch_num=0, max_retries=5):
    """
    sentences: list of plain strings to classify
    """
    user_msg = "Classify each sentence below. Return only the numbered labels."
    for i, sent in enumerate(sentences, 1):
        user_msg += f"{i}. {sent}"

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                temperature=0,
                max_tokens=120
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            err = str(e)
            wait_match = re.search(r'try again in (\d+)m([\d.]+)s', err)
            if wait_match:
                wait_secs = int(wait_match.group(1)) * 60 + float(wait_match.group(2))
                print(f"  Rate limited. Waiting {wait_secs:.0f}s before retry {attempt+1}...")
                time.sleep(wait_secs + 2)
            else:
                print(f"  Retry {attempt+1}: {e}")
                time.sleep(5)

    return ""

# =========================
# PARSE LABELS
# =========================

def normalize_label(raw):
    raw = raw.lower().strip()
    if raw.startswith("act"):  return "Action"
    if raw.startswith("dec"):  return "Decision"
    if raw.startswith("prop"): return "Proposal"
    return "Discussion"

def parse_labels(output, batch_size):
    if not output:
        return ["Discussion"] * batch_size

    labels = []
    for line in output.split("\n"):
        match = re.match(r"(\d+)\s*[:.]?\s*(\w+)", line.strip())
        if match:
            labels.append(normalize_label(match.group(2)))

    while len(labels) < batch_size:
        labels.append("Discussion")

    return labels[:batch_size]

# =========================
# POST FIX RULES
# =========================

def post_fix(sentence, label):
    s = sentence.lower().strip()

    # PRIORITY 0: Questions directed at speaker → Discussion
    if re.search(r'^speaker_\d+\s+(how|are you|were you|did you|do you|what|when|why)', s):
        return "Discussion"

    # PRIORITY 1: Past-tense status updates → Discussion
    if re.search(r"\b(i sent|i did|i ran|i checked|i finished|i completed|i submitted|deployment started|email sent)\b", s):
        return "Discussion"

    # PRIORITY 2: Facilitation phrases → Discussion
    if re.search(r"^let'?s (start|talk|discuss|look|move|assign|now|hear|run|also)\b", s):
        return "Discussion"

    # PRIORITY 3: Reactions → Discussion
    if re.search(r"\b(that'?s (exactly|great|good|fine|correct|right)|sounds good|makes sense|fair enough|understood)\b", s):
        return "Discussion"

    # PRIORITY 4: Committed deadlines → Action
    if re.search(r'\b(done|ready|complete|scheduled|confirmed)\s+(in|by|for|at)\s+\w+', s):
        return "Action"

    # PRIORITY 5: "I'll/I will" only with a concrete verb → Action
    if re.search(
        r"\b(i'?ll|i will)\s+(send|review|update|set up|schedule|share|create|build|"
        r"prepare|write|submit|assign|confirm|call|email|reach out|follow up|"
        r"fix|deploy|test|present|complete|draft|coordinate|organize|notify|ping)\b", s):
        return "Action"

    # PRIORITY 6: Direct named-speaker assignments → Action
    if re.search(r'^speaker_\d+\s+(can you|please|own|confirm|identify|give me|let me know)\b', s):
        return "Action"

    # PRIORITY 7: Question-based task assignment → Action
    if re.search(r'\bcan you (confirm|check|review|verify|look at|evaluate|initiate|identify|send|own)\b', s):
        return "Action"

    # PRIORITY 8: Confirmed timeline/date → Decision
    if re.search(r'\b(timeline is|release date is|target is|next sync is|next meeting is)\b', s):
        return "Decision"

    # PRIORITY 9: Scoping limits → Decision
    if re.search(r"\b(may not support|won'?t support|not included in|out of scope for)\b", s):
        return "Decision"

    # PRIORITY 10: Deferral → Decision
    if re.search(r'\b(pushed|deferred|moved)\s+(to|into)\s+(v\d|next|the)\b', s):
        return "Decision"

    # PRIORITY 11: Explicit confirmed outcomes → Decision
    if re.search(r"\b(we have decided|it is decided|we agreed|officially confirmed|we confirmed|we will not)\b", s):
        return "Decision"

    # PRIORITY 12: Proposals
    if re.search(r"\b(we should|we could|i suggest|i recommend|maybe we|what if we|could we|let'?s consider|how about)\b", s):
        return "Proposal"

    # Trust the LLM for everything else
    return label

# =========================
# MAIN FUNCTION (IMPORTANT)
# =========================

def labeling_pipeline(clean_text, batch_size=10):
    """
    Input:
        clean_text  (string from preprocessing)
        batch_size  (int) sentences per LLM call — keep at 10

    Output:
        list of dicts: [{speaker, sentence, label}, ...]
    """

    lines = clean_text.split("\n")
    data = []

    for line in lines:
        if ":" in line:
            spk, sent = line.split(":", 1)
            data.append((spk.strip(), sent.strip()))

    results = []
    total_batches = (len(data) + batch_size - 1) // batch_size

    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        batch_num = i // batch_size + 1
        print(f"  Labelling batch {batch_num}/{total_batches}...")

        sentences = [sent for (_, sent) in batch]
        raw = classify_batch(sentences, batch_num)
        labels = parse_labels(raw, len(batch))

        for (spk, sent), lbl in zip(batch, labels):
            lbl = post_fix(sent, lbl)
            results.append({
                "speaker": spk,
                "sentence": sent,
                "label": lbl
            })

        time.sleep(0.3)

    return results


In [9]:
#=========================================EXTRACTION============================================

import re

# =========================
# CLEAN TASK
# =========================

def clean_task(text):

    t = text.strip()

    t = re.sub(r'^Speaker_\d+\s*[, ]*\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^Speaker_\d+\s+(can you|could you|please)\s+', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^(can you|could you|will you|would you|please)\s+', '', t, flags=re.IGNORECASE)

    t = re.sub(r"^I\s+(will|can)\s+", '', t, flags=re.IGNORECASE)
    t = re.sub(r"^I['’]?ll\s+", '', t, flags=re.IGNORECASE)

    t = re.sub(r"^let's\s+", '', t, flags=re.IGNORECASE)
    t = re.sub(r'^we\s+(need to|should|must|have to)\s+', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^first,\s*', '', t, flags=re.IGNORECASE)

    t = re.sub(r'[?.!,;]+$', '', t)
    t = re.sub(r'\s+', ' ', t).strip()

    if t:
        t = t[0].upper() + t[1:]

    return t


# =========================
# ASSIGNEE
# =========================

def extract_assignee(text, speaker):

    match = re.match(r'^(Speaker_\d+)\s+(can you|please|will you)', text, re.IGNORECASE)
    if match:
        return match.group(1)

    if re.search(r"\b(I will|I['’]?ll|I can)\b", text, re.IGNORECASE):
        return speaker

    match = re.search(r'(Speaker_\d+)', text)
    if match:
        return match.group(1)

    return ""


# =========================
# DEADLINE
# =========================

def extract_deadline(text):

    t = text.lower()

    if "today" in t:
        return "today"
    if "tomorrow" in t:
        return "tomorrow"
    if "next week" in t:
        return "next week"

    match = re.search(r'in (\d+) (day|week|month)s?', t)
    if match:
        return match.group(0)

    return ""


# =========================
# DECISION
# =========================

def extract_decision(text):

    d = text.strip()

    d = re.sub(r'^Speaker_\d+\s+', '', d)
    d = re.sub(r"^let's\s+", '', d, flags=re.IGNORECASE)
    d = re.sub(r'^and\s+', '', d, flags=re.IGNORECASE)
    d = re.sub(r'[?.!,;]+$', '', d)

    if d:
        d = d[0].upper() + d[1:]

    return d


# =========================
# MAIN FUNCTION (IMPORTANT)
# =========================

def extraction_pipeline(labeled_data):
    """
    Input:
        labeled_data (list of dicts from labeling)

    Output:
        extracted_data (list of dicts)
    """

    results = []

    for row in labeled_data:

        sentence = row["sentence"]
        label = row["label"]
        speaker = row["speaker"]

        if label not in ["Action", "Decision"]:
            continue

        assignee = ""
        task = ""
        deadline = ""
        decision_text = ""

        if label == "Action":
            assignee = extract_assignee(sentence, speaker)
            deadline = extract_deadline(sentence)
            task = clean_task(sentence)

        else:
            decision_text = extract_decision(sentence)

        results.append({
            "sentence": sentence,
            "label": label,
            "speaker": speaker,
            "assignee": assignee,
            "task": task,
            "deadline": deadline,
            "decision_text": decision_text
        })

    return results

In [10]:
# =======================================================PRIORITIZER==========================================
import re
import time

# NOTE: reuses `client` (Groq) already initialised in the labeling cell above.

# =========================
# KEYWORDS
# =========================

urgency_keywords     = ["today", "asap", "immediately", "urgent"]
near_deadline_keywords = ["this week", "tomorrow"]
decision_keywords    = ["policy", "mandatory", "confirmed"]
action_keywords      = ["send", "create", "design", "build", "review", "clear", "circulate"]

vague_words = [
    "help", "progress", "look into", "try", "discuss",
    "clear", "sort", "handle", "follow up", "check", "do it", "get on it"
]
reference_triggers = ["those", "these", "that", "them", "it", "they", "this"]

# =========================
# VAGUE CHECK
# =========================

def is_vague(text):
    text = text.lower().strip()
    return any(k in text for k in vague_words) or any(f" {k} " in f" {text} " for k in reference_triggers)

# =========================
# GROQ CLARIFICATION
# =========================

def resolve_vague_with_groq(item, transcript_lines, current_idx, max_retries=2):
    """
    Uses Groq (llama-3.3-70b) to clarify vague action/decision text
    using surrounding transcript context.
    Falls back to original text on any error — never blocks the pipeline.
    """
    text = (item.get("task", "") + " " + item.get("decision_text", "")).strip()

    if not is_vague(text):
        return item

    # Use up to 6 lines of context immediately before this item
    context_window = transcript_lines[max(0, current_idx - 6): current_idx]
    context_str = "\n".join(context_window) if context_window else "(no prior context)"

    prompt = (
        f"You are clarifying a vague sentence from a meeting transcript.\n\n"
        f"Recent transcript context:\n{context_str}\n\n"
        f"Vague sentence:\n{text}\n\n"
        f"Rewrite it as one clear, specific sentence using the context above. "
        f"Do not add extra explanation. Return only the rewritten sentence."
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=80
            )
            clarified = response.choices[0].message.content.strip()

            # Sanity check: reject if response is empty or suspiciously long
            if not clarified or len(clarified) > 300:
                break

            if item["label"].lower() == "action":
                item["task"] = clarified
            else:
                item["decision_text"] = clarified

            return item

        except Exception as e:
            print(f"  ⚠️ Groq clarification skipped (attempt {attempt+1}): {e}")
            time.sleep(1)

    return item  # keep original text if all attempts fail

# =========================
# SCORING
# =========================

def calculate_score(item):
    score = 0

    text     = (item.get("task", "") + " " + item.get("decision_text", "")).lower()
    deadline = item.get("deadline", "").lower()
    assignee = item.get("assignee", "")

    if any(k in deadline for k in urgency_keywords):      score += 5
    elif any(k in deadline for k in near_deadline_keywords): score += 3
    if any(k in text for k in decision_keywords):         score += 4
    if any(k in text for k in action_keywords):           score += 2
    if assignee:                                          score += 2
    if is_vague(text):                                    score -= 2
    if not assignee:                                      score -= 1
    if not deadline:                                      score -= 1

    return score

# =========================
# MAIN FUNCTION (IMPORTANT)
# =========================

def prioritizer_pipeline(extracted_data, clean_text):
    """
    Input:
        extracted_data → list of dicts from extraction_pipeline
        clean_text     → full cleaned transcript string

    Output:
        {
          "top_actions":   [...],   # up to 3, sorted by priority score
          "top_decisions": [...]    # up to 3, sorted by priority score
        }
    """

    transcript_lines = clean_text.split("\n")
    processed = []
    total = len(extracted_data)

    for i, item in enumerate(extracted_data, 1):
        text_preview = (item.get("task", "") + item.get("decision_text", ""))[:50]
        print(f"  ⚡ Prioritizing item {i}/{total}: {text_preview}...")

        item  = resolve_vague_with_groq(item, transcript_lines, i - 1)
        score = calculate_score(item)
        item["priority_score"] = score
        processed.append(item)

    actions   = sorted(
        [x for x in processed if x["label"].lower() == "action"],
        key=lambda x: x["priority_score"], reverse=True
    )
    decisions = sorted(
        [x for x in processed if x["label"].lower() == "decision"],
        key=lambda x: x["priority_score"], reverse=True
    )

    return {
        "top_actions":   actions[:3],
        "top_decisions": decisions[:3]
    }

In [11]:
#=============================================Format&PDF==============================================

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle

PRIMARY_BLUE = colors.HexColor("#2F5D8C")
LIGHT_BLUE = colors.HexColor("#D9E2F3")

header_style = ParagraphStyle(
    name="Header",
    fontName="Times-Bold",
    fontSize=20,
    textColor=PRIMARY_BLUE,
    spaceAfter=10
)

section_style = ParagraphStyle(
    name="Section",
    fontName="Times-Bold",
    fontSize=15,
    textColor=PRIMARY_BLUE,
    spaceAfter=10
)

normal_style = ParagraphStyle(
    name="Normal",
    fontName="Times-Roman",
    fontSize=14,
    leading=18,
    spaceAfter=4
)

bullet_style = ParagraphStyle(
    name="Bullet",
    fontName="Times-Roman",
    fontSize=14,
    leading=18,
    leftIndent=10,
    spaceAfter=6
)

def horizontal_line(width=500):
    line = Table([[""]], colWidths=[width], rowHeights=[2])
    line.setStyle(TableStyle([
        ("LINEBELOW", (0, 0), (-1, -1), 0.5, colors.lightgrey)
    ]))
    return line


# =========================
# MAIN FUNCTION
# =========================

def format_pdf_pipeline(prioritized_output, meeting_details):

    pdf_path = f"/content/meeting_{meeting_details.get('title','output')}.pdf"

    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=letter,
        leftMargin=40,
        rightMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    elements = []

    # =========================
    # HEADER SECTION
    # =========================
    elements.append(Paragraph(f"<b>Meeting Title:</b> {meeting_details.get('title','')}", header_style))
    elements.append(Paragraph(f"<b>Date & Time:</b> {meeting_details.get('meeting_datetime','')}", normal_style))
    elements.append(Paragraph(f"<b>Location:</b> {meeting_details.get('meeting_mode','')}", normal_style))
    elements.append(Paragraph(f"<b>Facilitator:</b> {meeting_details.get('facilitator','')}", normal_style))
    elements.append(Paragraph(f"<b>Note Taker:</b> {meeting_details.get('note_taker','')}", normal_style))

    elements.append(Spacer(1, 8))
    elements.append(horizontal_line())
    elements.append(Spacer(1, 8))

    # =========================
    # ATTENDEES SECTION
    # =========================
    elements.append(Paragraph("Attendees", section_style))

    attendees = meeting_details.get("attendees", "")
    roles = meeting_details.get("roles", "")

    att_list = attendees.split(",") if attendees else []
    role_list = roles.split(",") if roles else []

    table_data = [["Name", "Role"]]

    for i in range(len(att_list)):
        name = att_list[i].strip()
        role = role_list[i].strip() if i < len(role_list) else ""
        table_data.append([name, role])

    att_table = Table(table_data, colWidths=[200, 200])

    att_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), LIGHT_BLUE),
        ("FONTNAME", (0, 0), (-1, 0), "Times-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ]))

    elements.append(att_table)

    elements.append(Spacer(1, 8))
    elements.append(horizontal_line())
    elements.append(Spacer(1, 8))

    # =========================
    # DECISIONS SECTION
    # =========================
    elements.append(Paragraph("Decisions Made", section_style))

    for d in prioritized_output["top_decisions"]:
        elements.append(Paragraph(f"• {d.get('decision_text','')}", bullet_style))

    elements.append(Spacer(1, 8))
    elements.append(horizontal_line())
    elements.append(Spacer(1, 8))

    # =========================
    # ACTION ITEMS
    # =========================
    elements.append(Paragraph("Action Items", section_style))

    table_data = [["Task", "Assigned To", "Deadline"]]

    for a in prioritized_output["top_actions"]:
        table_data.append([
            Paragraph(a.get("task",""), normal_style),
            Paragraph(a.get("assignee","-"), normal_style),
            Paragraph(a.get("deadline","-"), normal_style)
        ])

    table = Table(table_data, colWidths=[300, 120, 120])

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), LIGHT_BLUE),
        ("FONTNAME", (0, 0), (-1, 0), "Times-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))

    elements.append(table)

    elements.append(Spacer(1, 8))
    elements.append(horizontal_line())
    elements.append(Spacer(1, 8))

    # =========================
    # FOOTER
    # =========================
    elements.append(Paragraph("Ending Note", section_style))
    elements.append(Paragraph(f"<b>Meeting End Time:</b> {meeting_details.get('end_time','')}", normal_style))
    elements.append(Paragraph(f"<b>Notes:</b> {meeting_details.get('notes','')}", normal_style))

    # =========================
    # BUILD PDF
    # =========================
    doc.build(elements)

    return pdf_path

In [12]:
#========================================WHOLE PIPELINE======================================

def process_pipeline(audio_path, meeting_details):

    # 1. STT
    print("\n🎙️ [1/6] Starting Speech-to-Text (Whisper + Diarization)...")
    transcript = stt_pipeline(audio_path)
    print("✅ [1/6] STT complete.")

    # 2. Preprocessing
    print("\n🧹 [2/6] Preprocessing transcript...")
    clean_text = preprocess_pipeline(transcript)
    print("✅ [2/6] Preprocessing complete.")

    # 3. Labelling
    print("\n🏷️  [3/6] Labelling sentences (Action / Decision / Proposal / Discussion)...")
    labeled = labeling_pipeline(clean_text)
    print(f"✅ [3/6] Labelling complete. {len(labeled)} sentences labelled.")

    # 4. Extraction
    print("\n🔍 [4/6] Extracting actions and decisions...")
    extracted = extraction_pipeline(labeled)
    print(f"✅ [4/6] Extraction complete. {len(extracted)} items extracted.")

    # 5. Prioritizer
    print("\n⚡ [5/6] Prioritizing and resolving vague items...")
    prioritized = prioritizer_pipeline(extracted, clean_text)
    print("✅ [5/6] Prioritization complete.")

    # 6. Format & PDF
    print("\n📄 [6/6] Generating PDF...")
    pdf_path = format_pdf_pipeline(prioritized, meeting_details)
    print(f"✅ [6/6] PDF generated: {pdf_path}")
    print("\n🎉 Pipeline complete!\n")

    return {
        "transcript": transcript,
        "pdf_path": pdf_path
    }

In [ ]:
#========================================== Colab API ===========================================
from flask import Flask, request, jsonify
from flask_cors import CORS
import os
import json
import base64
import uuid
import threading

app = Flask(__name__)
CORS(app)  # allow browser requests from any origin

# In-memory job store: { job_id: {"status": "processing"|"done"|"error", ...} }
jobs = {}

# IMPORTANT: make sure process_pipeline is defined ABOVE this cell

# ── Route 1: Browser uploads audio directly here ─────────────────────────────
@app.route('/upload-audio', methods=['POST'])
def upload_audio():
    try:
        file = request.files.get('audio')
        if not file:
            return jsonify({'success': False, 'message': 'No file received'})

        filename  = f"{uuid.uuid4()}_{file.filename}"
        save_path = f"/content/{filename}"
        file.save(save_path)

        return jsonify({'success': True, 'file_id': save_path})
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'success': False, 'message': str(e)})


# ── Background pipeline worker ────────────────────────────────────────────────
def _run_pipeline(job_id, audio_path, meeting_details):
    try:
        result   = process_pipeline(audio_path, meeting_details)
        pdf_path = result["pdf_path"]
        with open(pdf_path, "rb") as pdf_file:
            pdf_base64 = base64.b64encode(pdf_file.read()).decode("utf-8")
        jobs[job_id] = {
            "status":     "done",
            "transcript": result["transcript"],
            "pdf_base64": pdf_base64
        }
        print(f"\n✅ Job {job_id} done\n")
    except Exception as e:
        import traceback
        traceback.print_exc()
        jobs[job_id] = {"status": "error", "error": str(e)}


# ── Route 2: webapp.py triggers processing — returns instantly ────────────────
@app.route('/process-meeting', methods=['POST'])
def process_meeting():
    try:
        payload         = request.get_json(force=True)
        audio_filename  = payload["audio_filename"]
        meeting_details = payload["meeting_details"]

        job_id = str(uuid.uuid4())
        jobs[job_id] = {"status": "processing"}

        # Run pipeline in background — response returns in milliseconds
        threading.Thread(
            target=_run_pipeline,
            args=(job_id, audio_filename, meeting_details),
            daemon=True
        ).start()

        return jsonify({"success": True, "job_id": job_id})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"success": False, "error": str(e)})


# ── Route 3: webapp.py polls this until done ──────────────────────────────────
@app.route('/job-status/<job_id>', methods=['GET'])
def job_status(job_id):
    job = jobs.get(job_id)
    if not job:
        return jsonify({"success": False, "error": "Job not found"}), 404
    return jsonify({"success": True, **job})


# ── START SERVER WITH CLOUDFLARE TUNNEL ──────────────────────────────────────
import subprocess, re, requests as req_lib

# 1. Download cloudflared if not already present
if not os.path.exists("/content/cloudflared"):
    os.system("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /content/cloudflared")
    os.system("chmod +x /content/cloudflared")

# 2. Start cloudflared and capture the public URL
cf_url = None

def start_cloudflare():
    global cf_url
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:5000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in proc.stdout:
        line = line.decode()
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            cf_url = match.group(0)
            print(f"\n✅ Colab API: {cf_url}")
            print("👉 Paste this URL into webapp.py or use /update-colab-url\n")
            try:
                resp = req_lib.post(
                    "http://10.238.77.27:5000/update-colab-url",
                    json={"url": cf_url},
                    timeout=5
                )
                if resp.ok:
                    print("✅ webapp.py updated automatically")
            except Exception:
                pass
            break

threading.Thread(target=start_cloudflare, daemon=True).start()

# 3. Run Flask
app.run(port=5000, threaded=True)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



✅ Colab API: https://strikes-pavilion-compatibility-admissions.trycloudflare.com
👉 Paste this URL into webapp.py or use /update-colab-url



INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 11:38:29] "POST /upload-audio HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 11:39:26] "POST /process-meeting HTTP/1.1" 200 -



🎙️ [1/6] Starting Speech-to-Text (Whisper + Diarization)...


...

INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 11:48:58] "GET /job-status/b79bb9c7-bbbb-4b72-9555-f58bcaebc9c3 HTTP/1.1" 200 -


  Labelling batch 17/17...


INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 11:49:14] "GET /job-status/b79bb9c7-bbbb-4b72-9555-f58bcaebc9c3 HTTP/1.1" 200 -


✅ [3/6] Labelling complete. 168 sentences labelled.

🔍 [4/6] Extracting actions and decisions...
✅ [4/6] Extraction complete. 15 items extracted.

⚡ [5/6] Prioritizing and resolving vague items...
  ⚡ Prioritizing item 1/15: Will have time to do some revision and some additi...
  ⚡ Prioritizing item 2/15: Asking you to send me the...
  ⚡ Prioritizing item 3/15: Okay, then I'm also going to present a little...
  ⚡ Prioritizing item 4/15: So far so we can make a point that here is ontolog...
  ⚡ Prioritizing item 5/15: The generation I put halfway done...
  ⚡ Prioritizing item 6/15: And Faye is doing the synthesis stuff as we speak...
  ⚡ Prioritizing item 7/15: Talk about our problems with the rephrasing and ho...
  ⚡ Prioritizing item 8/15: I'm not going to put in the figures from this...
  ⚡ Prioritizing item 9/15: Srini was gonna send me some slides, but he didn't...
  ⚡ Prioritizing item 10/15: Okay, I can give you a more recent if you want wel...
  ⚡ Prioritizing item 11/15: Copy a

INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 11:49:30] "GET /job-status/b79bb9c7-bbbb-4b72-9555-f58bcaebc9c3 HTTP/1.1" 200 -
